In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"

In [ ]:
import torch
from torchvision.models import ViT_B_16_Weights, vit_b_16
from training_utils import get_data_loaders
from types import SimpleNamespace
import matplotlib.pyplot as plt
from tqdm import tqdm
from utils import *
import torch.nn.functional as F

In [3]:
params = SimpleNamespace(
    gpu_idx=1,
    img_size=224,
    batch_size=16,
    output_dir="attention_maps/attention_maps_raw_vitb16_pretrained"
)

In [4]:
print("Number of visible GPUs :", torch.cuda.device_count())

torch.cuda.set_device(params.gpu_idx)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}:{torch.cuda.current_device()} {torch.cuda.get_device_name(torch.cuda.current_device())}" if torch.cuda.is_available() else "CPU")

Number of visible GPUs : 3
Device: cuda:1 NVIDIA GeForce RTX 2080 Ti


In [5]:
weights = ViT_B_16_Weights.IMAGENET1K_V1
model = vit_b_16(weights=weights).to(device)
model.eval();

In [6]:
train_loader, val_loader = get_data_loaders(batch_size=params.batch_size, persistent_workers=False, shuffle_val=True)

imgs, labels = next(iter(val_loader))
tensor = imgs[0].unsqueeze(0).to(device)  # add batch dimension and move to device

# create a mask for filtering indices that are not present in Imagenet100
present_indices = torch.tensor([
117,  70,  88, 133,   5,  97,  42,  60,  14,   3,
130,  55,  26,   0,  89, 127,  36,  67, 110,  65,
123,  57,  22,  21,   1,  71,  99,  16,  19, 108,
    18,  35, 124,  90,  74, 129, 125,   2,  64,  92,
138,  48,  54,  39,  56,  96,  84,  73,  77,  52,
    20, 118, 111,  59, 106,  75, 143,  80, 140,  11,
113,   4,  28,  50,  38, 104,  24, 107, 100,  81,
    94,  41,  68,   8,  66, 146,  29,  32, 137,  33,
141, 134,  78, 150,  76,  61, 112,  83, 144,  91,
135, 116,  72,  34,   6, 119,  46, 115,  93,   7
], device=device).sort().values

In [7]:
def get_accuracy(model, loader, topk=(1, 5)):
    correct_topk = {k: 0 for k in topk}
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(device), labels.to(device)
            labels = present_indices[labels]  # map labels to their corresponding indices in the full ImageNet
            outputs = model(images)

            # Top-k prédictions
            maxk = max(topk)
            _, pred = outputs.topk(maxk, dim=1)  # shape: [batch, maxk]

            total += labels.size(0)

            for k in topk:
                # Vérifie si la vraie classe est dans les k prédictions
                correct = pred[:, :k].eq(labels.view(-1, 1)).sum().item()
                correct_topk[k] += correct

    accs = {f"acc@{k}": correct_topk[k] / total for k in topk}
    return accs

In [8]:
val_acc = get_accuracy(model, val_loader, topk=(1, 5))
print(f"Top-1 Accuracy: {val_acc['acc@1']:.4f} Top-5 Accuracy: {val_acc['acc@5']:.4f}")

  0%|          | 0/313 [00:00<?, ?it/s]

Top-1 Accuracy: 0.8590 Top-5 Accuracy: 0.9734


In [9]:
# Nettoyer les anciens hooks
for _, module in model.named_modules():
    if hasattr(module, "_forward_pre_hooks"):
        module._forward_pre_hooks.clear()

qk_by_layer = {}
attn_by_layer = {}


def make_attention_hook(name):
    def hook(module, inputs):
        x = inputs[0]  # [B, T, C]
        B, T, C = x.shape
        H = module.num_heads
        D = C // H

        qkv = F.linear(x, module.in_proj_weight, module.in_proj_bias)
        q, k, _ = qkv.chunk(3, dim=-1)

        q = q.reshape(B, T, H, D).transpose(1, 2)  # [B, H, T, D]
        k = k.reshape(B, T, H, D).transpose(1, 2)  # [B, H, T, D]

        qk = (q @ k.transpose(-2, -1)) * (D ** -0.5)
        attn = torch.softmax(qk, dim=-1)

        qk_by_layer[name] = qk[0].detach().cpu()
        attn_by_layer[name] = attn[0].detach().cpu()

    return hook


layer_names = []
for name, module in model.named_modules():
    if name.endswith("self_attention"):
        module.register_forward_pre_hook(make_attention_hook(name))
        layer_names.append(name)

with torch.no_grad():
    _ = model(tensor)


layer_names = sorted(layer_names, key=lambda x: int(x.split('encoder_layer_')[1].split('.')[0]))

print(f"Nombre de layers capturés: {len(layer_names)}")
print("Exemple shape attn [heads, tokens, tokens]:", attn_by_layer[layer_names[0]].shape)

Nombre de layers capturés: 12
Exemple shape attn [heads, tokens, tokens]: torch.Size([12, 197, 197])


In [10]:
# Visualiser les cartes d'attention: 12 heads, 6 par ligne, pour chaque layer
os.makedirs(params.output_dir, exist_ok=True)

for layer_idx, name in enumerate(layer_names):
    attn = attn_by_layer[name]  # [H, T, T]
    n_heads = min(12, attn.shape[0])

    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    fig.suptitle(f"Layer {layer_idx} ({name}) - softmax(QK^T)", fontsize=13)

    for head in range(12):
        row, col = divmod(head, 6)
        ax = axes[row, col]

        if head < n_heads:
            attn_h = attn[head]
            ax.imshow(attn_h.numpy(), cmap="viridis", vmin=0.0, vmax=attn_h.max().item())
            ax.set_title(f"h{head} max={attn_h.max().item():.3f}", fontsize=9)
            ax.axis("off")
        else:
            ax.axis("off")

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(f"{params.output_dir}/layer_{layer_idx:02d}_attention_heads.png")
    plt.close(fig)